# KhaledDSML Research Studio — 0.10.0.dev0
## الاعتدالية متعددة المتغيرات وتحليل حساسية الفقد | Multivariate normality and missing-data sensitivity

**النسخة الحالية على PyPI: 0.9.0**. هذا الدفتر يعرض وظائف فرع تطوير **غير منشور**. جميع النتائج محُسوبة فعليًا؛ عدم رفض اختبار الاعتدالية لا يعني إثباتها.

**Published stable: 0.9.0.** New methods below require the unreleased development branch. Do not treat hypothetical MNAR shifts as causal evidence or inferential p-values.

In [ ]:
%pip install -q "git+https://github.com/noble2013/KhaledDSML.git@feature/v0.10-catalog-henze-zirkler"

### 1 — Henze–Zirkler | اختبار هينز–زيركلر
استقلالية المشاهدات وملاءمة نموذج التوزيع الطبيعي متعدد المتغيرات شرطان منفصلان. يستخدم الاختبار إحصائية HZ ومعايرة مونت كارلو. 
Independent, complete observations and full-rank covariance are required; Monte Carlo p-value resolution is limited by the number of replicates.

In [ ]:
import numpy as np
import pandas as pd
from khaled_dsml import (
    henze_zirkler_normality_test, mnar_delta_sensitivity,
    display_analysis_dashboard,
)

rng = np.random.default_rng(2026)
normal_sample = rng.normal(size=(100, 3))
hz = henze_zirkler_normality_test(
    normal_sample, n_resamples=199, random_state=17
)
display_analysis_dashboard(hz, theme="aurora")

### 2 — اختبار بديل لا يتبع التوزيع الطبيعي | Non-Gaussian alternative
بيانات مولدة عشوائياً من توزيع أسي؛ هذا مثال تعليمي لا يمثل دراسة سريرية أو اقتصادية حقيقية.
Exponential synthetic data; no claim about a real population.

In [ ]:
exponential_sample = rng.exponential(size=(110, 3))
hz_non_gaussian = henze_zirkler_normality_test(
    exponential_sample, n_resamples=199, random_state=17
)
display_analysis_dashboard(hz_non_gaussian, theme="ocean")

### 3 — MNAR delta sensitivity | تحليل حساسية دلتا
يغيّر هذا السيناريو القيم المستكملة فقط، ولا يثبت نوع آلية الفقد. لا تُحسب منه p-value أو فواصل ثقة.
The deltas are hypothetical imputed-only shifts, **not** evidence that MNAR holds or an identified probability model.

In [ ]:
study = pd.DataFrame({
    "income": [80., 95., np.nan, 125., 160., np.nan, 175., 192.],
    "age": [22., 27., 31., 35., 40., 45., 49., 55.]
})
completed_a = study.fillna({"income": 110.})
completed_b = study.fillna({"income": 140.})
sensitivity = mnar_delta_sensitivity(
    study, [completed_a, completed_b],
    column="income", deltas=[-45., -20., 0., 20., 45.]
)
display_analysis_dashboard(sensitivity, theme="amethyst")

### 4 — Interpretation / تفسير النتائج
- **العربية:** القيمة الاحتمالية للاعتدالية تعتمد على عدد المحاكاة. تحليل دلتا وصفي، ويجب تحديد سيناريوهات دلتا قبل رؤية النتائج قدر الإمكان.
- **English:** Monte Carlo simulation count sets p-value granularity. MNAR delta changes are scenario assumptions, not empirically estimated mechanisms.
- **Research integrity:** Do not fit preprocessing on test data, claim normality from non-rejection, or report uncomputed confidence intervals.